# Janelas complementares do piloto

**Papel nesta entrega:** Suplementar: visualizacao posterior do piloto

**Origem:** Revisao_piloto_Alda1_4_locais.zip / Complemento_visual_piloto_Alda1.ipynb

Copia de publicacao organizada em 7 out. 2026. Saidas, contadores e metadados de sessao foram removidos; isso nao significa que a campanha original nao tenha sido executada. Nenhum resultado novo foi produzido na curadoria. O registro de proveniencia relaciona codigo, fonte e alteracoes de configuracao.

Reapresenta TIFFs ja adquiridos. Nao altera intensidades dos arquivos, segmentacao ou escores. O painel USC associado a Figura 7 foi arquivado na versao aprovada; as observacoes humanas posteriores prevalecem sobre a impressao inicial do piloto.

# Piloto Alda-1: visualizar detalhes sem modificar os dados

Este complemento reutiliza os TIFFs ja baixados. Nao treina modelos, nao altera escores, nao abre R4 e nao preenche a ficha humana. Mantem os paineis originais e gera outra visualizacao para Mito em MEDINA e USC, onde aproximadamente 13,7% dos pixels estavam acima da faixa dos DMSO.

O codigo abaixo foi executado localmente com os seis TIFFs reais e conferido por hashes. A sessao Colab nao foi executada pelo assistente. Apenas NumPy, pandas e Pillow sao utilizados; nao ha atualizacao de dependencias.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive
drive.mount("/content/drive")
ORIGEM = Path("/content/drive/MyDrive/mba_tcc/comparacao_4_locais_HepG2/inspecao_biologica_4_locais/resultados/inspecao_20261003T200334_744222Z")
if not (ORIGEM / "imagens_baixadas.csv").is_file():
    raise FileNotFoundError("A pasta do piloto nao foi encontrada. Confira ORIGEM.")
DESTINO = ORIGEM.parent / (ORIGEM.name + "_janela_complementar_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ"))
print("Entrada preservada:", ORIGEM)
print("Nova saida:", DESTINO)

## Por que duas janelas?

Pixels acima do limite de exibicao ficam brancos no painel, mas podem ter valores diferentes no TIFF. Para recuperar detalhes, a segunda janela tem limite superior igual ao maior P99,5 entre os tres campos, nunca inferior ao limite original. O limite inferior continua o dos DMSO. **A mesma janela e aplicada ao tratamento e aos dois controles.** Nao ha ajuste individual por imagem.

A janela complementar utiliza tambem a imagem tratada para decidir como mostrar os pixels; ela e pos-hoc, nao normalizacao biologica. Comparar os locais em brilho absoluto continua inadequado sem metadados de aquisicao. Um pixel no valor maximo uint16 nao tem seu valor original recuperado pelo ajuste.

In [ ]:
"""Supplementary linear display of existing pilot TIFFs, never an analysis filter.
Keeps the original DMSO display and writes a second common range in a new folder.
No downloads, model fitting, segmentation, or modification of the source files.
"""
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont


def digest(path):
    with Path(path).open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


def read_gray(path):
    with Image.open(path) as image:
        if getattr(image, 'n_frames', 1) != 1:
            raise ValueError('Multipage TIFF: choose the plane explicitly, not automatically.')
        array = np.array(image)
    if array.ndim != 2 or array.dtype != np.uint16:
        raise ValueError(f'Expected a 2D uint16 image: {path}, {array.shape}, {array.dtype}')
    return array


def common_upper(arrays, original_upper):
    """Same upper limit for every image in the group, not a per-image adjustment."""
    return max(float(original_upper), *(float(np.percentile(a, 99.5)) for a in arrays))


def display_image(array, lower, upper, crop=False):
    if upper <= lower:
        raise ValueError('Invalid display range.')
    if crop:
        hh, ww = min(512, array.shape[0]), min(512, array.shape[1])
        yy, xx = (array.shape[0]-hh)//2, (array.shape[1]-ww)//2
        array = array[yy:yy+hh, xx:xx+ww]
    image = Image.fromarray((255*np.clip((array.astype(float)-lower)/(upper-lower), 0, 1)).astype('uint8'))
    # Rendering only. Never use this 8-bit derivative to recompute features.
    return image.resize((512, 512), Image.Resampling.LANCZOS)


def run_review(source, destination, raw_directory=None, sites=('MEDINA', 'USC'), channels=('Mito',)):
    source, destination = Path(source), Path(destination)
    source = source.resolve()
    if destination.resolve() == source:
        raise ValueError('Use a separate output folder.')
    destination.mkdir(parents=True, exist_ok=False)
    manifest_path = source/'imagens_baixadas.csv'
    scales_path = source/'escalas_exibicao_DMSO.csv'
    input_hashes = {str(p): digest(p) for p in (manifest_path, scales_path)}
    manifest = pd.read_csv(manifest_path)
    scales = pd.read_csv(scales_path)
    wanted = manifest[manifest.site.isin(sites) & manifest.canal.isin(channels)].copy()
    if wanted.empty or set(wanted.site) != set(sites) or not wanted.replica.eq('R1').all():
        raise ValueError('This review expects the listed sites and pilot R1 only.')
    if wanted.duplicated(['site','placa','replica','poco','campo','canal']).any():
        raise ValueError('Duplicate image keys.')
    raw_directory = Path(raw_directory) if raw_directory is not None else source/'imagens_originais'
    records=[]; panels=[]
    try:
        font=ImageFont.truetype('DejaVuSans.ttf', 19)
        small=ImageFont.truetype('DejaVuSans.ttf', 15)
    except OSError:
        font=ImageFont.load_default(size=19); small=ImageFont.load_default(size=15)
    for (site, plate, rep, field, channel), group in wanted.groupby(['site','placa','replica','campo','canal']):
        if len(group) != 3 or group.composto.eq('DMSO').sum() != 2:
            raise ValueError('The pilot expects one treatment and two controls per group.')
        group=group.assign(iscase=group.composto.ne('DMSO')).sort_values(['iscase','poco'], ascending=[False,True])
        arrays={}
        for _, row in group.iterrows():
            path=raw_directory/Path(row.arquivo_local).name
            h=digest(path)
            if h != row.sha256:
                raise ValueError(f'Original TIFF hash mismatch: {path.name}')
            input_hashes[str(path)]=h
            arrays[row.sample_id]=read_gray(path)
        matching=scales[(scales.site==site)&(scales.placa==plate)&(scales.replica==rep)&(scales.canal==channel)]
        if len(matching) != 1:
            raise ValueError('Missing or ambiguous original scale.')
        lower=float(matching.iloc[0].min_exibicao); old=float(matching.iloc[0].max_exibicao)
        upper=common_upper(arrays.values(), old)
        for crop in [False, True]:
            tile=512; gap=20; head=120; rowheight=tile+85
            canvas=Image.new('RGB',(3*tile+4*gap, head+2*rowheight+25),'white')
            draw=ImageDraw.Draw(canvas)
            title=f'{site} | {plate} {rep} campo {field} | {channel} | '+('recorte central 512 px' if crop else 'campo inteiro reduzido')
            draw.text((20,10), title,fill='black',font=font)
            draw.text((20,40),'Escala unica por linha para os tres pocos. Sem ajuste individual. Sem escala fisica.',fill='black',font=small)
            draw.text((20,65),'Linha 1: DMSO original. Linha 2: faixa ampliada comum. TIFFs e escores preservados.',fill='black',font=small)
            draw.text((20,88),'Limite superior complementar: maior P99,5 entre os tres campos (nunca menor que o anterior).',fill='black',font=small)
            for rr, high in enumerate([old,upper]):
                for cc, (_, row) in enumerate(group.iterrows()):
                    array=arrays[row.sample_id];xx=gap+cc*(tile+gap);yy=head+rr*rowheight
                    name='DMSO' if row.composto=='DMSO' else str(row.nome)
                    draw.text((xx,yy),f'{name} | {row.poco}',fill='black',font=font)
                    draw.text((xx,yy+27),f'Faixa {lower:.0f}-{high:.0f}; acima: {100*np.mean(array>high):.3f}% do campo',fill='black',font=small)
                    canvas.paste(display_image(array,lower,high,crop),(xx,yy+55))
            path=destination/f'{site}_{channel}_duas_faixas_{"crop" if crop else "inteiro"}.png'
            canvas.save(path); panels.append(str(path))
        for _,row in group.iterrows():
            array=arrays[row.sample_id]
            records.append(dict(site=site,poco=row.poco,sample_id=row.sample_id,composto=row.composto,canal=channel,
                min_exibicao=lower,max_original=old,max_suplementar=upper,
                clip_original_pct=100*float(np.mean(array>old)),clip_suplementar_pct=100*float(np.mean(array>upper)),
                pixels_65535=int(np.sum(array==65535)),max_dtype_pct=100*float(np.mean(array==65535)),
                n_pixels=array.size,max_valor=int(array.max()),sha256=row.sha256,
                nota='Pixel no teto do arquivo nao identifica a causa da saturacao; sem calibracao do detector.'))
    for path, h in input_hashes.items():
        if digest(path)!=h: raise RuntimeError('An input changed during rendering.')
    table=pd.DataFrame(records)
    table.to_csv(destination/'auditoria_TIFFs_e_faixas.csv',index=False)
    (destination/'proveniencia.json').write_text(json.dumps(dict(
        data_utc=datetime.now(timezone.utc).isoformat(),entradas_sha256=input_hashes,
        regra='Mesma faixa linear por grupo. Inferior = limite DMSO original. Superior = max(limite DMSO, maior P99.5 por campo).',
        selecao='MEDINA e USC Mito: diagnostico do clipping no piloto. Nao seleciona candidatos.',
        parametros_pos_hoc=True,pixels_originais_alterados=False,avaliacao_humana_preenchida=False,
        R4_lida=False,n_tiffs=len(table),n_paineis=len(panels)),ensure_ascii=False,indent=2),encoding='utf-8')
    return table,panels



In [ ]:
from IPython.display import display, Image as DisplayImage
TABELA, PAINEIS = run_review(ORIGEM, DESTINO)
display(TABELA)
for painel in PAINEIS:
    display(DisplayImage(filename=painel))
print("CONCLUIDO:", DESTINO)
print("Imagens, escores e ficha humana originais preservados.")

## Como ler

Em cada painel, a primeira linha mostra a escala original dos DMSO. A segunda linha amplia a faixa igualmente nos tres pocos; os controles ficam mais escuros por essa razao. Observe se ha detalhes recuperados, mas nao converta maior brilho em maior funcao ou massa mitocondrial. Cada campo representa um unico poco em R1, e o recorte nao e uma segunda amostra independente.

A interpretacao final deve ser registrada pelo pesquisador na ficha humana, mantendo as observacoes do assistente separadas. Para ampliar a inspecao a outros campos ou replicas, utilize o caderno de inspecao com um plano explicito, sem alterar a selecao de candidatos.